# **Hatespeech Detection** | 2025

# **I. Data Understanding**

# ***A. Penjabaran Studi Kasus***

Kamu harus bikin program deteksi ujaran kebencian (hate speech) dari teks tweet (data sosial media), dari awal banget:

a. Tahap I – Data Preprocessing
Membersihkan dan menyiapkan teks.

b. Tahap II – Pembobotan Kata (TF-IDF)
Mengubah teks jadi angka.

c. Tahap III – Klasifikasi (KNN)
Melatih model untuk mengenali kategori.

d. Tahap IV – Solusi (GUI)
Membuat tampilan sederhana untuk input teks baru dan tampilkan hasil klasifikasinya.

- Dataset berisi tweet dan label (“Netral”, “Ras”, “Agama”).

- Tujuan: mengenali apakah teks mengandung ujaran kebencian (berdasarkan ras atau agama) atau netral.

# ***B. Import Libraries***

In [239]:
import pandas as pd
import numpy as np
import tkinter as tk

# ***C. Import Dataset***

In [240]:
dataset = pd.read_csv("TABEL DATA LATIH HATESPEECH RISET.csv", sep=';', header=None)
dataset

,0,1,2
0,xndedlin,@amyliarm lingkunganmu keknya punya pemahaman ...,Netral
1,__succiduous,Udah jelek brengsek pula,Ras
2,KemenagMempawah,Lucunya penghuni negeri ini selalu di hiasi da...,Agama
3,1stKOREANguy1,"Yang jelek + miskin udah pasti bukan Kristen ,...",Ras
4,newsutdofficial,@MurtadhaOne1 Mereka memanfaatkan kebodohan ka...,Agama
...,...,...,...
2123,NaN,NaN,NaN
2124,NaN,NaN,NaN
2125,NaN,NaN,NaN
2126,NaN,NaN,NaN


# **II. Data Preparation**

# ***A. Exploratory Data Analysis***

### 1. Inspeksi Daftar Variabel

bikin nama variabelnya juga

In [241]:
dataset.columns = ['username', 'text', 'label']

### 2. Inspeksi Informasi Ringkas

In [242]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2128 entries, 0 to 2127
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   username  2001 non-null   object
 1   text      2001 non-null   object
 2   label     2000 non-null   object
dtypes: object(3)
memory usage: 50.0+ KB


### 3. Inspeksi Statistik Deskriptif

### 4. Inspeksi Data NaN

In [243]:
dataset.isna().sum()

username    127
text        127
label       128
dtype: int64

In [244]:
dataset = dataset.dropna(subset=['text', 'label']).reset_index(drop=True)

### 5. Inspeksi Data Duplicated

In [245]:
dataset.duplicated().sum()

np.int64(43)

# ***B. Text Preprocessing***

Tahap ini sesuai instruksi Tahap I di slide (cleaning, case folding, tokenizing, normalisasi, stemming, remove stopword).

### 1. Text Cleaning 

buang simbol, URL, angka, dan tanda baca

In [246]:
def cleaning_text(text):
    text = str(text)

    # Hapus URL & mention
    text = ' '.join([t for t in text.split() if not t.startswith('http')])
    text = ' '.join([t for t in text.split() if not t.startswith('@')])

    # Hapus simbol & angka
    clean = ''
    for char in text:
        if (char >= 'A' and char <= 'Z') or (char >= 'a' and char <= 'z') or char == ' ':
            clean += char

    return clean

dataset['clean'] = dataset['text'].apply(cleaning_text)

### 2. Case Folding

ubah jadi huruf kecil semua

In [247]:
dataset['clean'] = dataset['clean'].str.lower()

### 3. Tokenizing

pecah teks jd daftar kata

In [248]:
dataset['tokens'] = dataset['clean'].str.split()

### 4. Text Normalize + Stemming

ganti singkatan jd ngga singkat wkwkoaw

In [249]:
kamus = {
    'gak': 'tidak',
    'nggak': 'tidak',
    'ga': 'tidak',
    'aja': 'saja',
    'kek': 'seperti',
    'yg': 'yang',
    'dr': 'dari',
    'tdk': 'tidak',
    'tp': 'tapi',
    'dlm': 'dalam',
    'bgt': 'banget',
    'skrg': 'sekarang',
    'trs': 'terus',
    'udh': 'sudah',
    'lg': 'lagi',
    'sm': 'sama',
    'dgn': 'dengan',
    'kl': 'kalau',
    'jg': 'juga',
    'aja': 'saja'
}

In [250]:
stopwords = [
    'yang', 'dan', 'di', 'ke', 'dari', 'itu', 'ini', 'untuk', 'pada',
    'dengan', 'karena', 'agar', 'adalah', 'saja', 'atau', 'dalam',
    'tidak', 'iya', 'ya', 'aku', 'kamu', 'dia', 'mereka', 'kami',
    'saya', 'kau', 'loh', 'dong', 'deh', 'nih', 'tuh', 'pun', 'lah',
    'kan', 'ya', 'aja', 'udah', 'kok', 'tau'
]

In [251]:
def normalize_and_stem(tokens):
    hasil = []
    for kata in tokens:
        kata = kamus[kata] if kata in kamus else kata
        if len(kata) > 4:
            if kata.endswith('nya') or kata.endswith('lah') or kata.endswith('kan'):
                kata = kata[:-3]
        if kata not in stopwords:
            hasil.append(kata)
    return hasil

dataset['tokens'] = dataset['tokens'].apply(lambda x: x if isinstance(x, list) else [])
dataset['tokens'] = dataset['tokens'].apply(normalize_and_stem)

# Hapus baris yang hasil token-nya kosong
dataset = dataset[dataset['tokens'].apply(lambda x: len(x) > 0)].reset_index(drop=True)

# ***III. Pembobotan Kata***

### 1. Daftar Semua Kata Unik

In [252]:
vocab = []
for row in dataset['tokens']:
    for word in row:
        if word not in vocab:
            vocab.append(word)
vocab = np.array(vocab)

### 2. Term Frequency

In [253]:
def term_frequency(tokens):
    vec = np.zeros(len(vocab))
    if not tokens:
        return vec
    for word in tokens:
        if word in vocab:
            idx = np.where(vocab == word)[0][0]
            vec[idx] += 1
    vec = vec / len(tokens)
    return vec

tf_matrix = np.array([term_frequency(tokens) for tokens in dataset['tokens']])

### 3. Inverse Document Frequency

In [254]:
N = len(dataset)
df = np.zeros(len(vocab))
for i, word in enumerate(vocab):
    count = 0
    for tokens in dataset['tokens']:
        if word in tokens:
            count += 1
    df[i] = count

idf = np.log((N + 1) / (df + 1)) + 1

### 4. TF-IDF

In [255]:
tfidf_matrix = tf_matrix * idf
tfidf_matrix = tfidf_matrix / np.linalg.norm(tfidf_matrix, axis=1, keepdims=True)
tfidf_matrix = np.nan_to_num(tfidf_matrix)

In [256]:
word_counts = np.sum(tf_matrix > 0, axis=0)
mask = word_counts > 1
tfidf_matrix = tfidf_matrix[:, mask]
vocab = vocab[mask]

print(f"Jumlah kata setelah filter: {len(vocab)}")

Jumlah kata setelah filter: 2276


# ***IV. Modeling***

### 1. Splitting Dataset

In [257]:
X = tfidf_matrix
label_map = {"Netral" : 0, "Ras" : 1, "Agama" : 2}
y = np.array(dataset['label'].map(label_map))

In [258]:
label_map = {"Netral": 0, "Ras": 1, "Agama": 2}
y = np.array(dataset['label'].map(label_map))
idx = np.random.permutation(len(tfidf_matrix))
X = tfidf_matrix[idx]
y = y[idx]

split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

### 2. Model Class

In [259]:
class KNNManual:
    def __init__(self, k=5):
        self.k = k
        self.X_train = None
        self.y_train = None

    def fit(self, X_train, y_train):
        self.X_train = X_train
        self.y_train = y_train

    def euclidean_distance(self, x1, x2):
        return np.sqrt(np.sum((x1 - x2) ** 2))

    def predict_one(self, x):
        distances = np.array([self.euclidean_distance(x, x_train) for x_train in self.X_train])
        k_idx = distances.argsort()[:self.k]
        k_labels = self.y_train[k_idx]
        values, counts = np.unique(k_labels, return_counts=True)
        return values[np.argmax(counts)]

    def predict(self, X_test):
        return np.array([self.predict_one(x) for x in X_test])

    def evaluate(self, X_test, y_test):
        y_pred = self.predict(X_test)
        accuracy = np.sum(y_pred == y_test) / len(y_test)
        print(f"\nAkurasi Model: {round(accuracy * 100, 2)}%")
        return y_pred, accuracy

In [260]:
for k in [3, 5, 7, 9]:
    print(f"\n=== Evaluasi k={k} ===")
    model = KNNManual(k=k)
    model.fit(X_train, y_train)
    y_pred, acc = model.evaluate(X_test, y_test)


=== Evaluasi k=3 ===

Akurasi Model: 53.0%

=== Evaluasi k=5 ===

Akurasi Model: 51.0%

=== Evaluasi k=7 ===

Akurasi Model: 50.0%

=== Evaluasi k=9 ===

Akurasi Model: 49.0%


In [261]:
model = KNNManual(k=5)
model.fit(X_train, y_train)

# ***V. GUI***

In [262]:
def preprocess_input(teks):
    teks = cleaning_text(teks.lower())
    tokens = teks.split()
    tokens = normalize_and_stem(tokens)

    vec = np.zeros(len(vocab))
    for word in tokens:
        if word in vocab:
            idx = np.where(vocab == word)[0][0]
            vec[idx] += 1
    if len(tokens) > 0:
        vec = vec / len(tokens)
    vec = vec * idf[mask]       
    norm = np.linalg.norm(vec)
    if norm != 0:
        vec = vec / norm
    return vec

def predict_text():
    teks = entry_text.get("1.0", "end-1c").strip()
    if teks == "":
        tk.messagebox.showwarning("Peringatan", "Masukkan teks terlebih dahulu!")
        return

    vec = preprocess_input(teks)
    pred = model.predict(np.array([vec]))[0]

    label_map_inv = {0: "Netral", 1: "Ras", 2: "Agama"}
    hasil = label_map_inv[pred]

    label_result.config(text="Hasil Prediksi: " + hasil, fg="green")

root = tk.Tk()
root.title("Deteksi Ujaran Kebencian (Hate Speech Detector)")
root.geometry("500x350")
root.configure(bg="#222831")

label_title = tk.Label(root, text="Hate Speech Detection",
                       font=("Arial", 16, "bold"), bg="#222831", fg="#FFD369")
label_title.pack(pady=10)

label_instruksi = tk.Label(root, text="Masukkan teks/tweet di bawah ini:",
                           font=("Arial", 11), bg="#222831", fg="white")
label_instruksi.pack()

entry_text = tk.Text(root, height=7, width=50, font=("Arial", 10))
entry_text.pack(pady=10)

btn_predict = tk.Button(root, text="Prediksi", command=predict_text,
                        font=("Arial", 11, "bold"), bg="#FFD369", fg="#222831")
btn_predict.pack(pady=10)

label_result = tk.Label(root, text="Hasil Prediksi: -",
                        font=("Arial", 12, "bold"), bg="#222831", fg="white")
label_result.pack(pady=10)

label_footer = tk.Label(root, text="By ABEL | SMK 8 Malang 2025",
                        font=("Arial", 9), bg="#222831", fg="#888")
label_footer.pack(side="bottom", pady=10)

root.mainloop()
